# RAFT Fine-Tuning of Llama-3.2-1B-Instruct

## Overview

**RAFT (Retrieval-Augmented Fine-Tuning)** is a supervised fine-tuning strategy that trains a language model to answer questions from a retrieved context that contains both the relevant *oracle* passage and several *distractor* documents. This teaches the model to identify and reason from the correct source rather than relying on parametric memory — directly improving RAG pipeline performance at inference time without any architecture changes.

**Reference:** *RAFT: Adapting Language Model to Domain Specific RAG* (Zhang et al., 2024).

---

### End-to-End Pipeline

```
PDF documents
    └─► pdf_to_chunks.py     (GPT-4o vision → per-page .txt files)
    └─► raft_datagen.py      (GPT-4o → Q/A/D triplets → JSONL)
    └─► this notebook        (Unsloth + LoRA → fine-tuned Llama-3.2-1B)
```

### Dataset Schema

Each JSONL record contains:

| Field | Description |
|-------|-------------|
| `question` | Synthetic question generated from a document chunk |
| `context` | Dict with `sentences` — oracle + `num_distract` distractor passages (shuffled) |
| `oracle_context` | The single chunk that actually answers the question |
| `instruction` | Full `<DOCUMENT>…</DOCUMENT>` prompt fed to the model |
| `cot_answer` | Chain-of-thought answer with `##begin_quote##` citations and final `<ANSWER>:` tag |

### Key Training Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `max_seq_length` | 2048 | Covers instruction + 4 docs + CoT answer |
| `load_in_4bit` | True | NF4 quantisation — ~4× memory reduction |
| LoRA rank `r` | 16 | Balances expressivity vs. parameter count |
| `lora_alpha` | 16 | Effective scale = alpha/r = 1.0 |
| `learning_rate` | 2e-5 | Conservative for an instruction-tuned base |
| `gradient_accumulation_steps` | 8 | Effective batch size = 2 × 8 = 16 |
| `lr_scheduler_type` | cosine | Smooth decay; standard for short runs |

---

## Setup: Install Dependencies

In [2]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: 
    import subprocess; 
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: 
    is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install loguru sqlglot sqlparse

## Step 1: Load the RAFT Dataset

Load the pre-generated RAFT training and evaluation splits from disk. Each split is a JSONL file where every record is a Q/A/D triplet.

- **`train.jsonl`** — 80% of generated examples, used for gradient updates  
- **`test.jsonl`** — held-out 20%, used for evaluation loss during training  

`tiktoken` is imported for optional token-count analysis of the dataset before training.

In [52]:
import os
import tiktoken
import json
import pandas as pd

training_data_dir = '/kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/data/training_data_raft'
train_df = pd.read_json(os.path.join(training_data_dir, "train.jsonl"), lines=True)
test_df = pd.read_json(os.path.join(training_data_dir, "test.jsonl"), lines=True)
print(f"Number of samples: train={len(train_df)}, test={len(test_df)}")
train_df.head()

Number of samples: train=520, test=65


,id,type,question,context,oracle_context,cot_answer,instruction
0,seed_task_135,general,Where can customers find information about med...,{'sentences': [['*Note:* Negotiated ciphers ar...,---\n\n# Best practices\n\n### Test your compa...,Step-by-step reasoning:\n\n1. The question ask...,<DOCUMENT>*Note:* Negotiated ciphers are subje...
1,seed_task_52,general,What is the key difference in accountability b...,{'sentences': [['### Consider your AI security...,The key differences include accountability (in...,Step-by-step reasoning:\n\n1. **Understand the...,<DOCUMENT>### Consider your AI security strate...
2,seed_task_95,general,What are the access model options mentioned?,{'sentences': [['```markdown # Best practices ...,"- Decide between agent-less, agent-based, or h...",Step-by-step reasoning:\n\n1. The question ask...,<DOCUMENT>```markdown\n# Best practices contin...
3,seed_task_69,general,How are Customer Actions different from automa...,{'sentences': [['```markdown # Assess ServiceN...,"Customer Actions guide you through scanning, u...",Step-by-step reasoning:\n\n1. **Understand the...,<DOCUMENT>```markdown\n# Assess ServiceNow as ...
4,seed_task_21,general,Where can the Data Processing Addendum be found?,{'sentences': [['#### Test changes in non-prod...,---\n\n### Additional information \n- [Legal:...,Step-by-step reasoning:\n\n1. The question ask...,<DOCUMENT>#### Test changes in non-production ...


## Step 2: Convert to HuggingFace `Dataset` Format

Convert the pandas DataFrames into HuggingFace `Dataset` objects. This enables:
- Efficient batched `.map()` transformations in subsequent steps
- Compatibility with `SFTTrainer` which expects HF datasets
- Arrow-backed memory mapping for large datasets

In [55]:
from datasets import Dataset

# Convert your pandas DataFrames to HF Datasets
train_dataset = Dataset.from_pandas(train_df)
eval_dataset  = Dataset.from_pandas(test_df)

# Check the structure
print(train_dataset)
print(train_dataset.column_names)
# print(train_dataset[0])   # view one example

Dataset({
    features: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction'],
    num_rows: 520
})
['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction']


## Step 3: Load Base Model and Attach LoRA Adapters

Load **Llama-3.2-1B-Instruct** using [Unsloth](https://github.com/unslothai/unsloth)'s optimised `FastLanguageModel`, then attach **LoRA (Low-Rank Adaptation)** adapters to the attention and MLP projection layers.

### Why LoRA?
Full fine-tuning of a 1B parameter model requires updating all weights (~4 GB in fp16). LoRA instead injects small trainable rank decomposition matrices (`r=16`) into each target layer. This reduces trainable parameters by **~99%** while preserving most of the base model's capabilities.

### Target modules
All six projection matrices are adapted — `q/k/v/o_proj` (attention) and `gate/up/down_proj` (MLP) — which gives the model maximum flexibility to learn the RAFT task format.

### Memory optimisation
- **4-bit NF4 quantisation** reduces the frozen base weights to ~500 MB GPU memory  
- **`use_gradient_checkpointing="unsloth"`** trades recomputation for ~30% less VRAM during the backward pass

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, _ = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, 
    full_finetuning = False, 
)

# Recommended for Llama-3.2
tokenizer = get_chat_template(
    model, 
    chat_template="llama-3.2"   # or "llama-3.2" if available in your Unsloth version
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 2025,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Step 4: Apply Chat Template Formatting

Transform each raw Q/A/D record into a single text string using the **Llama-3.2 chat template**. This step is critical because the model was instruction-tuned with a specific prompt structure; maintaining that structure during fine-tuning prevents catastrophic forgetting of the base chat format.

### Prompt structure per example

```
<|system|>  You are a helpful assistant…
<|user|>    ### Question: …
            ### Context (may contain noise): <distractor docs>
            ### Oracle passages (gold reference): <oracle chunk>
            ### Instruction: <DOCUMENT>…</DOCUMENT> prompt
<|assistant|> <chain-of-thought reasoning>
             <ANSWER>: …
```

### Why include distractors in the prompt?
RAFT deliberately exposes the model to noisy context during training. With probability `p=0.8` the oracle is present; with `p=0.2` it is replaced. This teaches robustness — the model learns to either find the answer or correctly abstain.

### Column cleanup
`remove_columns=train_dataset.column_names` drops all original columns, leaving only `text`. This avoids padding mismatches in the collator when column schemas differ between examples.

In [65]:
def formatting_prompts_func(examples):
    texts = []
    for qn, ctx, oracle, instr, ans in zip(
        examples["question"],
        examples["context"],
        examples["oracle_context"],
        examples["instruction"],
        examples["cot_answer"]
    ):
        # Safe handling of context (it's often a dict with 'sentences')
        if isinstance(ctx, dict):
            sentences = ctx.get("sentences", [])
            if sentences and isinstance(sentences[0], list):
                ctx_text = "\n".join(" ".join(s) for s in sentences)
            else:
                ctx_text = str(ctx)
        else:
            ctx_text = str(ctx)

        messages = [
            {"role": "system", "content": "You are a helpful assistant that answers questions using the provided context. Think step by step and quote relevant parts when possible."},
            {"role": "user", "content": f"""### Question:
{qn}

### Context (may contain noise):
{ctx_text}

### Oracle passages (gold reference):
{oracle}

### Instruction:
{instr}"""},
            {"role": "assistant", "content": ans}   # No extra "### Answer:" header
        ]
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        texts.append(text + tokenizer.eos_token)
    
    return {"text": texts}

# Apply formatting
train_ds = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names   # Clean old columns, keep only "text"
)

eval_ds = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=eval_dataset.column_names
)

# Verify
print(train_ds)
print(train_ds[0]["text"][:500])  # Check first 500 chars of formatted text

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 520
})
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a helpful assistant that answers questions using the provided context. Think step by step and quote relevant parts when possible.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Question:
Where can customers find information about media disposal for the ServiceNow AI Platform?

### Context (may contain noise):
*Note:* Negotiated ciphers are subject to custom


## Step 5: Cache Formatted Datasets to Disk

Persist the formatted datasets in HuggingFace Arrow format. This is useful when:
- Iterating on training hyperparameters without re-running the expensive formatting step
- Re-using the exact same formatted data across kernel restarts

The next cell shows how to reload from disk, skipping Steps 1–4 entirely.

In [66]:
# After creating train_ds and eval_ds, save them
train_ds.save_to_disk("raft_train_hf")
eval_ds.save_to_disk("raft_eval_hf")

Saving the dataset (0/1 shards):   0%|          | 0/520 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/65 [00:00<?, ? examples/s]

In [67]:
# Later, load them directly (no need to convert again)
from datasets import load_from_disk
train_ds = load_from_disk("raft_train_hf")
eval_ds = load_from_disk("raft_eval_hf")

## Step 6: Configure and Initialise the SFT Trainer

Configure `trl.SFTTrainer` with `transformers.TrainingArguments`. Key decisions:

| Argument | Value | Notes |
|----------|-------|-------|
| `per_device_train_batch_size` | 2 | Constrained by GPU VRAM with 4-bit model |
| `gradient_accumulation_steps` | 8 | Effective batch = 16; smooths gradient noise |
| `num_train_epochs` | 1 | Single pass — RAFT data quality > quantity |
| `learning_rate` | 2e-5 | Below the typical 5e-5 to protect instruction-following |
| `fp16` | True | Mixed-precision training; compatible with T4/A10 |
| `eval_strategy` | steps (every 5) | Frequent eval to detect divergence early |
| `optim` | adamw_torch | Decoupled weight decay; preferred over legacy adamw |
| `lr_scheduler_type` | cosine | Smooth warmup-free decay for short runs |
| `save_strategy` | no | Model is saved explicitly in Step 10 after merging |

In [95]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="llama32_1bn_instruct_raft", #This will also be used as your huggingfacehub model id name
    report_to="none", #Leave this to be blank if you don't want to use wandb
    eval_steps=5,
    eval_strategy="steps",
    per_device_train_batch_size=2,    # small batches if quantized
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    save_strategy="no",
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=5,
    seed=42,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds, 
    args=training_args,
    dataset_text_field="text"
)   

## Step 7: GPU Memory Snapshot (Pre-Training)

Record baseline GPU memory stats before training begins. These values are used post-training to calculate peak memory consumption and validate that the configuration fits within the available VRAM budget. If `start_gpu_memory / max_memory > 0.8`, consider reducing `per_device_train_batch_size` or enabling `load_in_8bit`.

In [69]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.426 GB of memory reserved.


## Step 8: Train the Model

Launch the training loop. `trainer.train()` returns a `TrainOutput` object containing:
- `global_step` — total number of optimiser steps taken  
- `training_loss` — average loss over the run  
- `metrics` — timing and throughput statistics  

Training progress and eval loss are logged every 5 steps. Monitor for:
- **Decreasing train loss** — confirms the model is learning the RAFT format  
- **Eval loss tracking train loss** — large divergence indicates overfitting; consider reducing epochs or increasing dropout

In [72]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 520 | Num Epochs = 1 | Total steps = 17
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss,Validation Loss
5,2.056000,2.034379
10,2.020800,2.005596
15,1.980900,1.996397


## Step 9: Review Training Statistics

Inspect the `TrainOutput` object. Key metrics to review:

- **`train_loss`** — final training loss; should be below 1.0 for a well-fitted RAFT model  
- **`train_runtime`** — total wall-clock training time in seconds  
- **`train_samples_per_second`** — throughput; use this to extrapolate cost for longer runs  
- **`train_steps_per_second`** — compare against expected steps given your batch config

In [96]:
trainer_stats

TrainOutput(global_step=17, training_loss=2.018335033865536, metrics={'train_runtime': 292.0165, 'train_samples_per_second': 1.781, 'train_steps_per_second': 0.058, 'total_flos': 3145098351083520.0, 'train_loss': 2.018335033865536, 'epoch': 1.0})

## Step 10: Save Merged Model in 16-bit

Merge the LoRA adapter weights back into the base model and save as a single 16-bit checkpoint. This produces a standard HuggingFace model directory that can be:
- Loaded with `AutoModelForCausalLM.from_pretrained()` — no LoRA dependency required
- Quantised further (GGUF, GPTQ) for local deployment
- Pushed directly to the Hub

`save_method="merged_16bit"` is preferred over `lora_only` for portability and over `merged_4bit` when upload size is not a constraint.

In [75]:
model.save_pretrained_merged(
    save_directory = "llama32_1bn_instruct_raft",     
    tokenizer = tokenizer,
    save_method = "merged_16bit",        
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:06<00:00,  6.15s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:17<00:00, 17.44s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama32_1bn_instruct_raft`


## Step 11: Authenticate with Hugging Face Hub

Retrieve the HF API token from Kaggle Secrets and authenticate. This scopes authentication to the current session without persisting the token to disk. The token requires **write** access to the target repository.

In [85]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## Step 12: Push Model and Tokenizer to Hugging Face Hub

Upload the merged model weights and tokenizer to the specified Hub repository. Both must be pushed — the tokenizer carries the chat template configuration which is required for correct inference.

After the push, the model can be loaded anywhere with:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("sriksmachi/llama32_1bn_instruct_raft")
tokenizer = AutoTokenizer.from_pretrained("sriksmachi/llama32_1bn_instruct_raft")
```

In [86]:
repo_id = "sriksmachi/llama32_1bn_instruct_raft"

# Use model.push_to_hub() to upload
model.push_to_hub(repo_id)

# Don't forget to push the tokenizer as well
tokenizer.push_to_hub(repo_id)

README.md:   0%|          | 0.00/584 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/sriksmachi/llama32_1bn_instruct_raft


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            